<a href="https://colab.research.google.com/github/francescopatane96/eNERVE/blob/main/2_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#2 Preprocessing 

1. remove duplicates

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('matches_final.csv', index_col=0).reset_index(drop=True)
display(df)

In [ ]:
df.count()

Entry Name               21051
Organism                 21051
Subcellular location     21051
Sequence                 21051
dtype: int64

In [ ]:
df_no_duplicates = df.drop_duplicates(subset='Sequence')

In [ ]:
df_no_duplicates.count()

Entry Name               20046
Organism                 20046
Subcellular location     20046
Sequence                 20046
dtype: int64

In [ ]:
df_no_duplicates.to_csv('matches_no_duplicates.csv')

In [ ]:
display(df.groupby('Subcellular location').count())

#df = df.groupby('Subcellular location')


In [ ]:
df_in = df[df['Subcellular location'] == 'i']
df_out = df[df['Subcellular location'] == 'o']
df_m = df[df['Subcellular location'] == 'm']
df_im = df[df['Subcellular location'] == 'im']

In [ ]:
df_in.to_csv('in.csv')
df_out.to_csv('out.csv')
df_m.to_csv('membrane.csv')
df_im.to_csv('internal_m.csv')

In [ ]:
!awk -F , '{print ">"$3"\n"$6}' matches_no_duplicates.csv > sequences_no_dupl.fasta.   #repeat for every class

2. CD-HIT

proteins clustering: we remove proteins using a similarity threshold = 25%

In [ ]:
!pip install Bio

In [ ]:
import sys 
from Bio import SeqIO 

from .fasta format to .csv

In [ ]:
#-----------------------------------------repeat for every .fasta file = for every class-----------------------------------------------
fasta_file = '025.fasta' 
ids = []
seq = []
for seq_record in SeqIO.parse(fasta_file, "fasta"): 
  ids.append(seq_record.id)
  seq.append(str(seq_record.seq))

In [ ]:
res = dict(zip(ids, seq))

In [ ]:
with open('025.csv', 'w') as csv_file:  
    writer = csv.writer(csv_file)
    for key, value in res.items():
       writer.writerow([key, value])

#------------------------------------------------------------end------------------------------------------------------------------------

join  all csv files, this is done using the index column 'Entry Name'

In [ ]:
df_in = pd.read_csv('in25.csv', names=['Entry', 'Sequence'], index_col='Entry')
df_in['Subcellular location'] = 'in'


df_in_m = pd.read_csv('intermem25.csv', names=['Entry', 'Sequence'], index_col='Entry')
df_in_m['Subcellular location'] = 'in_mem'

df_m = pd.read_csv('membrane25.csv', names=['Entry', 'Sequence'], index_col='Entry')
df_m['Subcellular location'] = 'mem'

df_out = pd.read_csv('out_25.csv', names=['Entry', 'Sequence'], index_col='Entry')
df_out['Subcellular location'] = 'out'

frames = [df_in, df_in_m, df_out, df_m]

df = pd.concat(frames)
print(df)
print(df_out.shape)
print(df_m.shape)

df.to_csv('merged_25_allprot.csv')

In [ ]:
#

In [ ]:
df = pd.read_csv('merged_25_allprot.csv', sep=',')   #upload csv obtained from notebook #1
display(df)

In [ ]:
#df.drop(columns=df.columns[0], axis=1, inplace=True)
#display(df)

In [ ]:
shuffled = df.sample(      #shuffles data (rows)
    frac=1,                ##returns entire df
    random_state=42         #makes the process reproducible
    
    ).reset_index(drop=True)

In [ ]:
display(shuffled)

In [ ]:
#indicates how many proteins belong to every class. to generate an equilibrate training dataset,
#we need to choose an equal amount of proteins for every class.
#choose the quantity of the class that has the lower count (secreted) --> x.sample() parameter
print(shuffled['Subcellular location'].value_counts())

In [ ]:
data = shuffled.groupby('Subcellular location').apply(lambda x: x.sample(201).reset_index(drop=True))
display(data)

In [ ]:
print(data['Subcellular location'].value_counts())

In [ ]:
full_dummies = pd.get_dummies(data['Subcellular location'])
#del full_dummies[full_dummies.columns[-1]]
full_matrix = pd.concat([data, full_dummies], axis=1)
del full_matrix['Subcellular location']

In [ ]:
display(full_matrix)

In [ ]:
#create the training set and the validation (held-out) set
from sklearn.model_selection import train_test_split

train, test = train_test_split(full_matrix, test_size=0.2, random_state=42)

In [ ]:
display(train)

In [ ]:
display(test)

In [ ]:
train.to_csv('train.csv')
test.to_csv('test.csv')

In [ ]:
#